In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))

import time
import cv2
import numpy as np
import keyboard
import pygame
import threading
import torch as th
import vgamepad as vg
import gymnasium as gym
from win32gui import GetWindowRect, GetClientRect
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.policies import ActorCriticPolicy
from resident_requiem import RERequiemEnv
from utils import get_last_index, LSTMWrapper

# --- CONFIG ---
DEVICE = th.device("cuda" if th.cuda.is_available() else "cpu")
SCALE = 2
SCREEN_WIDTH = 854
SCREEN_HEIGHT = 480
MAX_FPS = 30
train_path = "./imitation/"
sub_train_path = train_path + "steps"
current_epoch = 23 # 23 # 15 # 0
lstm_state = None 

train_path="./imitation/"
sub_train_path = train_path + "steps"

last_index_imitation = get_last_index(str(sub_train_path), "bc_policy", "zip")


class RendererThread(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.frame = None
        self.info = ""
        self.color = (0, 0, 255)
        self.fps = 0
        self.running = True
        self.current_epoch = 0
        self.lock = threading.Lock()

    def update_data(self, frame, info, color, fps, current_epoch):
        with self.lock:
            self.frame = frame
            self.info = info
            self.color = color
            self.fps = fps
            self.current_epoch = current_epoch

    def run(self):
        pygame.init()
        window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.HWSURFACE | pygame.DOUBLEBUF)
        font = pygame.font.SysFont("Arial", 22)
        
        while self.running:
            pygame.event.pump()
            
            with self.lock:
                if self.frame is not None:
                    img_view = cv2.resize(self.frame, (SCREEN_WIDTH, SCREEN_HEIGHT), interpolation=cv2.INTER_NEAREST)
                    img_rgb = cv2.cvtColor(img_view, cv2.COLOR_BGR2RGB)
                    
                    surf = pygame.surfarray.make_surface(img_rgb.swapaxes(0, 1))
                    window.blit(surf, (0, 0))
                    
                    txt = font.render(
                        f"{self.info} | FPS: {int(self.fps)} | current_epoch: {self.current_epoch}", True, (255, 255, 255)
                    )
                    pygame.draw.circle(window, self.color, (30, 30), 10)
                    window.blit(txt, (50, 20))
                    
                    pygame.display.flip()
            time.sleep(0.01)


policy = None

def main():
    global policy
    global current_epoch 
    myEnv = RERequiemEnv()
    env = WarpFrame(myEnv, width=128, height=128)
    env = DummyVecEnv([lambda: env])
    env = VecFrameStack(env, 4)

    renderer = RendererThread()
    renderer.start()

    obs = env.reset()
    ai_agent = None
    could_use_ai = False

    fps_avg_frame_count = 0
    fps_start_time = time.time()
    actual_fps = 0

    print("--- System started (Press K to use AI) ---")

    try:
        while True:
            loop_start = time.time()
            
            if keyboard.is_pressed('k'):
                could_use_ai = not could_use_ai
                if could_use_ai:
                    if current_epoch >= 24:
                        current_epoch += 3
                    else:
                        current_epoch += 1

                    if current_epoch > last_index_imitation:
                        current_epoch = last_index_imitation
        
                    # TODO: comment
                    # current_epoch = 33

                    # TODO: fix to last model on the step folder
                    if current_epoch > 38:
                        current_epoch = 38
                        
                    latest_model_path = sub_train_path + f"/bc_policy{current_epoch}.zip"
                    policy = ActorCriticPolicy.load(latest_model_path)


                    has_lstm = hasattr(policy, 'lstm') or hasattr(policy, 'features_extractor') and hasattr(policy.features_extractor, 'lstm')
            
                    if has_lstm:
                        policy = LSTMWrapper(policy)
                        policy.reset()
                        print("has lstm")
                else:
                    print("Manual mode activated")
                time.sleep(0.3)

            action = [np.zeros(18, dtype=np.int8)]
            if could_use_ai and policy:
                pred_act, _ = policy.predict(obs, deterministic=False)
                action = [pred_act]

            else:
                # Manual
                if keyboard.is_pressed('up'): action[0][0] = 1
                if keyboard.is_pressed('down'): action[0][1] = 1
                if keyboard.is_pressed('left'): action[0][2] = 1
                if keyboard.is_pressed('right'): action[0][3] = 1
                if keyboard.is_pressed('i'): action[0][4] = 1
                if keyboard.is_pressed('o'): action[0][5] = 1
                if keyboard.is_pressed('p'): action[0][6] = 1

            obs, _, _, _ = env.step(action)

            status = "AI ACTIVATED" if could_use_ai else "MANUAL"
            color = (0, 255, 0) if could_use_ai else (0, 0, 255)

            
            renderer.update_data(obs[0, :, :, -1], status, color, actual_fps, current_epoch)

            fps_avg_frame_count += 1
            if time.time() - fps_start_time >= 1.0: # update fps every 1 second
                actual_fps = fps_avg_frame_count / (time.time() - fps_start_time)
                fps_avg_frame_count = 0
                fps_start_time = time.time()

            # time_to_wait = (1/MAX_FPS) - (time.time() - loop_start)
            # if time_to_wait > 0:
            #     time.sleep(time_to_wait)

    except KeyboardInterrupt:
        pass
    finally:
        myEnv.close()
        renderer.running = False
        pygame.quit()

if __name__ == "__main__":
    main()

pygame 2.1.3 (SDL 2.0.22, Python 3.11.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
67414
Window founded HWND: 67414.
--- System started (Press K to use AI) ---


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)


has lstm
Manual mode activated
has lstm
Manual mode activated
has lstm
Manual mode activated
has lstm
Manual mode activated
has lstm
Manual mode activated
has lstm
Manual mode activated
